# 03 Pair Eligibility

Filter stable anti-persistent fits, estimate structural convergence horizons and select up to 40 pairs. A smaller valid population is reported without loosening thresholds.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Imports and saved run


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'research_config.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the Pairs_trading repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_session import NotebookSession

session = NotebookSession.active()
cfg = session.config
RUN_DIR = session.run
session.begin('03_eligibility', ['03_fou'])


## 2. Anti persistent and stable fits

Require 0 < H < 0.5, positive sigma/variance and daily Euler stability 0 < kappa < 2.


In [ ]:
from src.pair_eligibility import filter_antipersistent_pairs, compute_structural_t70, select_top_pairs_by_structural_t70
fou = session.frame('fractional_ou_parameters')
pool = filter_antipersistent_pairs(fou)
print(f'{len(pool)} of {len(fou)} fits pass model eligibility')
if pool.empty:
    raise ValueError('No stable anti-persistent fits.')
display(pool.head())


## 3. Structural convergence horizons

This is the expensive formation simulation. Its target, path count and seed are the settings saved in Module 01.


In [ ]:
structural = compute_structural_t70(pool, starting_z=cfg.entry_z, target_probability=cfg.target_probability,
    max_horizon_days=cfg.structural_horizon, n_paths=cfg.n_paths, seed=cfg.seed)
session.save('structural_results', structural)
display(structural[['pair', 'structural_t70', 'structural_probability_max']].head(15))


## 4. Rank and save the portfolio

The full finite-horizon pool is saved separately for matched placebo sampling.


In [ ]:
eligible_pool = structural.loc[structural.structural_t70.notna()].copy()
top_pairs = select_top_pairs_by_structural_t70(eligible_pool, cfg.top_n)
session.save('eligible_pool', eligible_pool)
session.save('eligible_pairs', top_pairs)
if top_pairs.empty:
    raise ValueError('No finite structural horizons.')
session.save('selected_horizon_summary', top_pairs.structural_t70.describe().to_frame('horizon'))
print(f'{len(top_pairs)} selected pairs from {len(eligible_pool)} eligible pairs')
display(top_pairs[['pair', 'hurst', 'structural_t70', 'structural_probability_max']])


## Save module completion

Wait for this confirmation before moving to the next notebook.


In [ ]:
session.finish('03_eligibility')
